### 축3

최종 축3 변수

1. EV 충전 수요 — 주차장 주변 or 자치구별 전기차 등록 밀도
2. 주차장 자체 전력 소비 규모 — 면수 +주차장운영시간

1. 주차장운영시간 조인하기(전처리데이터에 없음)

In [2]:
import pandas as pd

parking_raw = pd.read_csv(r'C:\Users\seonu\Documents\DR-project\NEW\dataset\서울시 공영주차장 안내 정보(원본).csv', encoding='cp949')
print(parking_raw.columns.tolist())

['주차장코드', '주차장명', '주소', '주차장 종류', '주차장 종류명', '운영구분', '운영구분명', '전화번호', '주차현황 정보 제공여부', '주차현황 정보 제공여부명', '총 주차면', '유무료구분', '유무료구분명', '야간무료개방여부', '야간무료개방여부명', '평일 운영 시작시각(HHMM)', '평일 운영 종료시각(HHMM)', '주말 운영 시작시각(HHMM)', '주말 운영 종료시각(HHMM)', '공휴일 운영 시작시각(HHMM)', '공휴일 운영 종료시각(HHMM)', '최종데이터 동기화 시간', '토요일 유,무료 구분', '토요일 유,무료 구분명', '공휴일 유,무료 구분', '공휴일 유,무료 구분명', '월 정기권 금액', '노상 주차장 관리그룹번호', '기본 주차 요금', '기본 주차 시간(분 단위)', '추가 단위 요금', '추가 단위 시간(분 단위)', '버스 기본 주차 요금', '버스 기본 주차 시간(분 단위)', '버스 추가 단위 시간(분 단위)', '버스 추가 단위 요금', '일 최대 요금', '위도', '경도']


In [11]:
parking_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 3696 entries, 0 to 3695
Data columns (total 40 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   주차장코드              3696 non-null   int64  
 1   주차장명               3696 non-null   str    
 2   주소                 3696 non-null   str    
 3   주차장 종류             3696 non-null   str    
 4   주차장 종류명            3696 non-null   str    
 5   운영구분               3696 non-null   int64  
 6   운영구분명              3696 non-null   str    
 7   전화번호               1670 non-null   str    
 8   주차현황 정보 제공여부       3696 non-null   int64  
 9   주차현황 정보 제공여부명      3696 non-null   str    
 10  총 주차면              3517 non-null   float64
 11  유무료구분              3696 non-null   str    
 12  유무료구분명             3696 non-null   str    
 13  야간무료개방여부           3696 non-null   str    
 14  야간무료개방여부명          3696 non-null   str    
 15  평일 운영 시작시각(HHMM)   3693 non-null   float64
 16  평일 운영 종료시각(HHMM)   3693 non-null   

In [4]:
print(parking_raw[['주차장명', '평일 운영 시작시각(HHMM)', '평일 운영 종료시각(HHMM)']].head(20).to_string())

                  주차장명  평일 운영 시작시각(HHMM)  평일 운영 종료시각(HHMM)
0        초안산근린공원주차장(구)             900.0            1900.0
1      마들스타디움(근린공원)(구)               0.0            2400.0
2     마장동(건물) 공영주차장(구)               0.0            2400.0
3          영등포여고 공영(구)               0.0            2400.0
4         당산근린공원 공영(구)               0.0            2400.0
5             대림운동장(구)               0.0            2400.0
6        휘경마을 공영주차장(구)               0.0            2400.0
7           중산공영주차장(구)             900.0            1900.0
8   방학동 도깨비시장 공영주차장(구)             900.0            2200.0
9            은평평화공원(구)               0.0            2400.0
10    압구정 428 공영주차장(구)             900.0            2100.0
11         신림동공영주차장(구)               0.0            2400.0
12       포이초교 공영주차장(구)               0.0            2400.0
13       수유2동 공영주차장(구)               0.0            2400.0
14     송중동(임시)공영주차장(구)               0.0            2400.0
15          바람개비주차장(구)             900.0            1900

In [5]:
# 전처리 완료 데이터와 합칠 컬럼 선택
cols_needed = ['주차장명', '총 주차면', 
               '평일 운영 시작시각(HHMM)', '평일 운영 종료시각(HHMM)',
               '주말 운영 시작시각(HHMM)', '주말 운영 종료시각(HHMM)',
               '공휴일 운영 시작시각(HHMM)', '공휴일 운영 종료시각(HHMM)']

parking_sub = parking_raw[cols_needed].copy()

# 평일 운영시간(시간) 계산
def calc_hours(start, end):
    try:
        s = int(str(int(start)).zfill(4)[:2])
        e = int(str(int(end)).zfill(4)[:2])
        if e == 24:  # 2400 = 자정 = 24시간으로 처리
            e = 24
        hours = e - s
        return hours if hours > 0 else 24  # 0이면 24시간 운영
    except:
        return None

parking_sub['평일운영시간'] = parking_raw.apply(
    lambda r: calc_hours(r['평일 운영 시작시각(HHMM)'], r['평일 운영 종료시각(HHMM)']), axis=1)
parking_sub['주말운영시간'] = parking_raw.apply(
    lambda r: calc_hours(r['주말 운영 시작시각(HHMM)'], r['주말 운영 종료시각(HHMM)']), axis=1)

print(parking_sub[['주차장명', '평일운영시간', '주말운영시간']].head(10).to_string())
print(parking_sub['평일운영시간'].value_counts().sort_index())

                 주차장명  평일운영시간  주말운영시간
0       초안산근린공원주차장(구)    10.0    10.0
1     마들스타디움(근린공원)(구)    24.0    24.0
2    마장동(건물) 공영주차장(구)    24.0    24.0
3         영등포여고 공영(구)    24.0    24.0
4        당산근린공원 공영(구)    24.0    24.0
5            대림운동장(구)    24.0    24.0
6       휘경마을 공영주차장(구)    24.0    24.0
7          중산공영주차장(구)    10.0    10.0
8  방학동 도깨비시장 공영주차장(구)    13.0    13.0
9           은평평화공원(구)    24.0    24.0
평일운영시간
1.0        2
7.0        3
8.0        6
9.0      505
10.0    1635
11.0     282
12.0      85
13.0     106
14.0      24
15.0      12
16.0       3
17.0       1
18.0       2
23.0      21
24.0    1006
Name: count, dtype: int64


In [6]:
parking_clean = pd.read_csv(r'C:\Users\seonu\Documents\DR-project\NEW\dataset\서울시_공영주차장_전처리완료.csv', encoding='utf-8-sig')

# 주차장명 기준 merge
parking_merged = parking_clean.merge(
    parking_sub[['주차장명', '총 주차면', '평일운영시간', '주말운영시간']],
    on='주차장명',
    how='left'
)

print(f"전체: {len(parking_merged)}개")
print(f"운영시간 결측: {parking_merged['평일운영시간'].isna().sum()}개")
print(parking_merged[['주차장명', '평일운영시간', '주말운영시간']].head(10).to_string())

전체: 1037개
운영시간 결측: 5개
                      주차장명  평일운영시간  주말운영시간
0  DDP동측(양쪽) 관광버스전용 주차장(시)    24.0    24.0
1  DDP동측(양쪽) 관광버스전용 주차장(시)    24.0    24.0
2  DDP동측(양쪽) 관광버스전용 주차장(시)    24.0    24.0
3  DDP동측(양쪽) 관광버스전용 주차장(시)    24.0    24.0
4  DDP동측(양쪽) 관광버스전용 주차장(시)    24.0    24.0
5  DDP동측(양쪽) 관광버스전용 주차장(시)    24.0    24.0
6  DDP동측(양쪽) 관광버스전용 주차장(시)    24.0    24.0
7  DDP동측(양쪽) 관광버스전용 주차장(시)    24.0    24.0
8  DDP동측(양쪽) 관광버스전용 주차장(시)    24.0    24.0
9  DDP동측(양쪽) 관광버스전용 주차장(시)    24.0    24.0


아 맞다 중복.....

In [7]:
# 중복 주차장명의 운영시간 다른 경우 확인
dup_names = parking_raw[parking_raw.duplicated(subset='주차장명', keep=False)]['주차장명'].unique()

for name in dup_names[:5]:  # 일단 5개만
    sub = parking_sub[parking_sub['주차장명'] == name][['주차장명', '평일운영시간', '주말운영시간']]
    if sub['평일운영시간'].nunique() > 1:
        print(f"[다름] {name}")
        print(sub.to_string())
        print()

[다름] 마들스타디움(근린공원)(구)
                 주차장명  평일운영시간  주말운영시간
1     마들스타디움(근린공원)(구)    24.0    24.0
3189  마들스타디움(근린공원)(구)    10.0     9.0

[다름] 불암산 공영주차장(구)
              주차장명  평일운영시간  주말운영시간
22    불암산 공영주차장(구)    24.0    24.0
3123  불암산 공영주차장(구)     9.0    24.0



왜 (구)가 있지

In [9]:
# 주차장명에서 (시) (구) 패턴 확인
import re
parking_raw['관할'] = parking_raw['주차장명'].str.extract(r'\((시|구)\)$')
print(parking_raw['관할'].value_counts(dropna=False))

관할
구    2057
시    1639
Name: count, dtype: int64


(구) 제거

In [10]:
# (시) 주차장만 필터링 후 중복 제거
parking_si = parking_sub[parking_sub['주차장명'].str.endswith('(시)')].drop_duplicates(subset='주차장명', keep='first')
print(f"(시) 주차장 수: {len(parking_si)}개")

# 중복 주차장명 운영시간 다른 경우 확인
for name in parking_si['주차장명'].unique():
    sub = parking_sub[parking_sub['주차장명'] == name]
    if len(sub) > 1 and sub['평일운영시간'].nunique() > 1:
        print(f"[다름] {name}")
        print(sub[['주차장명', '평일운영시간', '주말운영시간']].to_string())

(시) 주차장 수: 212개


In [12]:
print(f"원본 (시) 주차장: {len(parking_si)}개")
print(f"전처리 완료: {len(parking_clean)}개")

# 전처리 완료에 있는데 원본에 없는 것
our_names = set(parking_clean['주차장명'].tolist())
si_names = set(parking_si['주차장명'].tolist())

print(f"\n전처리에만 있는 것: {our_names - si_names}")
print(f"\n원본에만 있는 것 (일부): {list(si_names - our_names)[:10]}")

원본 (시) 주차장: 212개
전처리 완료: 100개

전처리에만 있는 것: {'방화역 공영주차장(시)', '청계5 공영주차장(시)', '청계3 공영주차장(시)', '청계6 공영주차장(시)', '청계2 공영주차장(시)'}

원본에만 있는 것 (일부): ['행운주차장(민영)(시)', '남산소파길 공영주차장(시)', '신천주차장(민영)(시)', '방화역(동) 공영주차장(시)', '여의서로 공영주차장(시)', '남대문시장2번게이트앞 이륜차 주차장(시)', '서울암사동유적 주차장(시)', '210-1 주차장(민영)(시)', '세종대로1 관광버스 승하차 허용 구간(시)', 'DDP 유어스빌딩 앞 관광버스 주차허용 구간(시)']


전처리를 뭐뭐 한거지...?
- 위, 경도 중앙값으로 중복 제거
- 주차장 이름 정규화(어떤 기준으로 정규화?)
- (구)/ (민영)(시) 제거
- 이륜차 전용, 승하차 구간 등 특수 목적 제거

그냥 전처리된 데이터셋 기준으로 조인

In [13]:
# 전처리 완료 주차장명 기준으로 원본에서 매칭
parking_sub_dedup = parking_sub.drop_duplicates(subset='주차장명', keep='first')

parking_merged = parking_clean.merge(
    parking_sub_dedup[['주차장명', '총 주차면', '평일운영시간', '주말운영시간']],
    on='주차장명',
    how='left'
)

print(f"조인 후: {len(parking_merged)}개")
print(f"운영시간 결측: {parking_merged['평일운영시간'].isna().sum()}개")
print(parking_merged[parking_merged['평일운영시간'].isna()]['주차장명'].tolist())

조인 후: 100개
운영시간 결측: 5개
['방화역 공영주차장(시)', '청계2 공영주차장(시)', '청계3 공영주차장(시)', '청계5 공영주차장(시)', '청계6 공영주차장(시)']


원본이랑 전처리완료랑 이름이 달라서 못찾음

In [14]:
# 결측 5개 원본에서 유사한 이름 찾기
missing = ['방화역 공영주차장(시)', '청계2 공영주차장(시)', '청계3 공영주차장(시)', '청계5 공영주차장(시)', '청계6 공영주차장(시)']

for name in missing:
    keyword = name.replace('공영주차장(시)', '').strip()
    matched = parking_sub[parking_sub['주차장명'].str.contains(keyword, na=False)]
    print(f"\n[{name}] → 유사 이름:")
    print(matched[['주차장명', '평일운영시간', '주말운영시간']].to_string())


[방화역 공영주차장(시)] → 유사 이름:
                 주차장명  평일운영시간  주말운영시간
1883    신방화역 공영주차장(시)    24.0    24.0
2612  방화역(서) 공영주차장(시)    13.0    13.0
2613  방화역(동) 공영주차장(시)    13.0    13.0

[청계2 공영주차장(시)] → 유사 이름:
                    주차장명  평일운영시간  주말운영시간
125     청계2(북2) 공영주차장(시)    10.0     6.0
126     청계2(북2) 공영주차장(시)    10.0     6.0
127     청계2(북2) 공영주차장(시)    10.0     6.0
128     청계2(북2) 공영주차장(시)    10.0     6.0
129     청계2(북2) 공영주차장(시)    10.0     6.0
130     청계2(북2) 공영주차장(시)    10.0     6.0
131     청계2(북2) 공영주차장(시)    10.0     6.0
132     청계2(북2) 공영주차장(시)    10.0     6.0
133     청계2(북2) 공영주차장(시)    10.0     6.0
667   청계2(돈화문로) 공영주차장(시)    10.0     6.0
668   청계2(돈화문로) 공영주차장(시)    10.0     6.0
669   청계2(돈화문로) 공영주차장(시)    10.0     6.0
670   청계2(돈화문로) 공영주차장(시)    10.0     6.0
671   청계2(돈화문로) 공영주차장(시)    10.0     6.0
672   청계2(돈화문로) 공영주차장(시)    10.0     6.0
673   청계2(돈화문로) 공영주차장(시)    10.0     6.0
674   청계2(돈화문로) 공영주차장(시)    10.0     6.0
675   청계2(돈화문로) 공영주차장(시)    10.0     6.0
676   청계2(돈화문로) 공영주

서/동 합치고

청계는 구역별로 나뉜거 합침

다 운영시간 같은데 청계5,6만 구역별로 운영시간이 다름

청계 5,6의 운영시간은 중앙값 이용

In [26]:
for keyword, our_name in [('청계5', '청계5 공영주차장(시)'), ('청계6', '청계6 공영주차장(시)')]:
    sub = parking_sub[parking_sub['주차장명'].str.contains(f'^{keyword}', na=False)]
    med_weekday = sub['평일운영시간'].median()
    med_weekend = sub['주말운영시간'].median()
    print(f"{our_name} → 평일중앙값: {med_weekday}h, 주말중앙값: {med_weekend}h")
    parking_merged.loc[parking_merged['주차장명'] == our_name, '평일운영시간'] = med_weekday
    parking_merged.loc[parking_merged['주차장명'] == our_name, '주말운영시간'] = med_weekend

manual_mapping = {
    '방화역 공영주차장(시)': '방화역(서) 공영주차장(시)',
    '청계2 공영주차장(시)': '청계2(북2) 공영주차장(시)',
    '청계3 공영주차장(시)': '청계3(북1) 공영주차장(시)',
}
for our_name, raw_name in manual_mapping.items():
    row = parking_sub[parking_sub['주차장명'] == raw_name][['평일운영시간', '주말운영시간']].iloc[0]
    parking_merged.loc[parking_merged['주차장명'] == our_name, '평일운영시간'] = row['평일운영시간']
    parking_merged.loc[parking_merged['주차장명'] == our_name, '주말운영시간'] = row['주말운영시간']

print(f"\n결측 남은 수: {parking_merged['평일운영시간'].isna().sum()}")

청계5 공영주차장(시) → 평일중앙값: 10.0h, 주말중앙값: 6.0h
청계6 공영주차장(시) → 평일중앙값: 24.0h, 주말중앙값: 6.0h

결측 남은 수: 0


In [27]:
parking_merged.to_csv(
    r'C:\Users\seonu\Documents\DR-project\NEW\OUTPUT\서울시_공영주차장_축3.csv',
    index=False,
    encoding='utf-8-sig'
)
print("저장 완료")
print(parking_merged[['주차장명', '총 주차면', '평일운영시간', '주말운영시간']].to_string())

저장 완료
                        주차장명   총 주차면  평일운영시간  주말운영시간
0    DDP동측(양쪽) 관광버스전용 주차장(시)     1.0    24.0    24.0
1            가마산고가밑 공영주차장(시)     1.0    10.0     6.0
2           가산동 금천교 공영주차장(시)     1.0    10.0     6.0
3              가양3동 공영주차장(시)    56.0    10.0     6.0
4             가양라이품 공영주차장(시)    38.0    24.0    24.0
5               가양역 공영주차장(시)    79.0    13.0    13.0
6              개화산역 공영주차장(시)   320.0    24.0    24.0
7               개화역 공영주차장(시)   483.0    24.0    24.0
8               관철동 공영주차장(시)     1.0    13.0     5.0
9          구로디지털단지역 공영주차장(시)    91.0    24.0    24.0
10             구파발역 공영주차장(시)   399.0    24.0    24.0
11           난지중앙로노상공영주차장(시)     1.0    13.0    13.0
12      남대문 시장 관광버스전용 주차장(시)     1.0    24.0    24.0
13      남대문 초입 관광버스전용 주차장(시)     1.0    24.0    24.0
14           남대문 화물 공영주차장(시)     1.0    10.0    10.0
15         남부여성발전센터 공영주차장(시)   128.0    24.0    24.0
16    남산공원 소월로 관광버스전용 주차장(시)     1.0    24.0    24.0
17    남산공원 소파로 관광버스전용 주차장(시)     1.0    

### 2. EV 충전 데이터 수집
    - 주차장 주변 EV 충전소 현황
    - 자치구별 전기차 등록 대수

#### - 전국전기차충전소표준데이터 (공공데이터포털) OPEN API요청

In [28]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time

API_KEY = 'c402314eb9d8ffed9740068d4f953034c92a943b239536adb11b7af21d796a8f'
BASE_URL = 'https://apis.data.go.kr/B552584/EvCharger/getChargerInfo'

def fetch_ev_chargers(sido='서울', page_no=1, num_of_rows=9999):
    params = {
        'serviceKey': API_KEY,
        'pageNo': page_no,
        'numOfRows': num_of_rows,
        'sido': sido,
    }
    response = requests.get(BASE_URL, params=params)
    return response.content

# 1페이지 먼저 테스트
xml_data = fetch_ev_chargers()
root = ET.fromstring(xml_data)

# 결과 확인
total_count = root.find('.//totalCount')
print(f"총 건수: {total_count.text if total_count is not None else '없음'}")
print(f"응답 코드: {root.find('.//resultCode').text}")
print(f"응답 메시지: {root.find('.//resultMsg').text}")

# 첫 번째 아이템 구조 확인
items = root.findall('.//item')
print(f"\n가져온 건수: {len(items)}")
if items:
    for child in items[0]:
        print(f"  {child.tag}: {child.text}")

총 건수: 502107
응답 코드: 00
응답 메시지: NORMAL SERVICE.

가져온 건수: 9999
  statNm: 낙성대동주민센터
  statId: ME174013
  chgerId: 01
  chgerType: 06
  addr: 서울특별시 관악구 낙성대로4가길 5
  addrDetail: null
  location: null
  lat: 37.476296
  lng: 126.9583876
  useTime: 24시간 이용가능
  busiId: ME
  bnm: 환경부
  busiNm: 환경부
  busiCall: 1661-9408
  stat: 2
  statUpdDt: 20260413201957
  lastTsdt: 20260412210330
  lastTedt: 20260412214330
  nowTsdt: None
  powerType: None
  output: 50
  method: 단독
  zcode: 11
  zscode: 11620
  kind: G0
  kindDetail: G003
  parkingFree: Y
  note: None
  limitYn: N
  limitDetail: None
  delYn: N
  delDetail: None
  trafficYn: N
  year: 2017
  floorNum: 1
  floorType: F
  maker: 시그넷


서울만 필터링!

In [29]:
# 전체 서울 데이터 수집 (9999개씩 페이지네이션)
all_items = []
page = 1

while True:
    xml_data = fetch_ev_chargers(page_no=page)
    root = ET.fromstring(xml_data)
    items = root.findall('.//item')
    
    if not items:
        break
    
    for item in items:
        data = {child.tag: child.text for child in item}
        # 서울만 필터링
        if data.get('addr', '') and '서울' in data.get('addr', ''):
            all_items.append(data)
    
    print(f"페이지 {page} 완료 - 누적 서울 충전소: {len(all_items)}개")
    
    total = int(root.find('.//totalCount').text)
    if page * 9999 >= total:
        break
    
    page += 1
    time.sleep(0.5)

df_ev = pd.DataFrame(all_items)
print(f"\n최종 서울 EV 충전소: {len(df_ev)}개")
print(df_ev[['statNm', 'addr', 'lat', 'lng']].head())

페이지 1 완료 - 누적 서울 충전소: 425개
페이지 2 완료 - 누적 서울 충전소: 2260개
페이지 3 완료 - 누적 서울 충전소: 3509개
페이지 4 완료 - 누적 서울 충전소: 4611개
페이지 5 완료 - 누적 서울 충전소: 6163개
페이지 6 완료 - 누적 서울 충전소: 6965개
페이지 7 완료 - 누적 서울 충전소: 8495개
페이지 8 완료 - 누적 서울 충전소: 9722개
페이지 9 완료 - 누적 서울 충전소: 11446개
페이지 10 완료 - 누적 서울 충전소: 12967개
페이지 11 완료 - 누적 서울 충전소: 14912개
페이지 12 완료 - 누적 서울 충전소: 16594개
페이지 13 완료 - 누적 서울 충전소: 17186개
페이지 14 완료 - 누적 서울 충전소: 18238개
페이지 15 완료 - 누적 서울 충전소: 18986개
페이지 16 완료 - 누적 서울 충전소: 19634개
페이지 17 완료 - 누적 서울 충전소: 21375개
페이지 18 완료 - 누적 서울 충전소: 22085개
페이지 19 완료 - 누적 서울 충전소: 24203개
페이지 20 완료 - 누적 서울 충전소: 25221개
페이지 21 완료 - 누적 서울 충전소: 26225개
페이지 22 완료 - 누적 서울 충전소: 27307개
페이지 23 완료 - 누적 서울 충전소: 28450개
페이지 24 완료 - 누적 서울 충전소: 29784개
페이지 25 완료 - 누적 서울 충전소: 31583개
페이지 26 완료 - 누적 서울 충전소: 33812개
페이지 27 완료 - 누적 서울 충전소: 35313개
페이지 28 완료 - 누적 서울 충전소: 39310개
페이지 29 완료 - 누적 서울 충전소: 40992개
페이지 30 완료 - 누적 서울 충전소: 41489개
페이지 31 완료 - 누적 서울 충전소: 43444개
페이지 32 완료 - 누적 서울 충전소: 44817개
페이지 33 완료 - 누적 서울 충전소: 46495개
페이지 34 완료 - 누적 서울 충전소: 48713

서울 EV충전소 저장

In [30]:
df_ev.to_csv(
    r'C:\Users\seonu\Documents\DR-project\NEW\OUTPUT\서울_EV충전소.csv',
    index=False,
    encoding='utf-8-sig'
)
print(f"저장 완료 - 총 {len(df_ev)}개")

저장 완료 - 총 74024개


주차장 좌표 기준 반경 500m 내 충전소 수 계산

In [31]:
from math import radians, sin, cos, sqrt, atan2

# 위도/경도 숫자 변환
df_ev['lat'] = pd.to_numeric(df_ev['lat'], errors='coerce')
df_ev['lng'] = pd.to_numeric(df_ev['lng'], errors='coerce')
df_ev = df_ev.dropna(subset=['lat', 'lng'])

# 하버사인 거리 계산 함수
def haversine(lat1, lng1, lat2, lng2):
    R = 6371000  # 지구 반경 (m)
    lat1, lng1, lat2, lng2 = map(radians, [lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlng/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# 주차장별 반경 500m 내 충전소 수 계산
ev_lat = df_ev['lat'].values
ev_lng = df_ev['lng'].values

results = []
for _, row in parking_merged.iterrows():
    count = sum(
        haversine(row['위도'], row['경도'], lat, lng) <= 500
        for lat, lng in zip(ev_lat, ev_lng)
    )
    results.append(count)

parking_merged['ev_충전소_500m'] = results
print(parking_merged[['주차장명', 'ev_충전소_500m']].sort_values('ev_충전소_500m', ascending=False).head(10).to_string())

                   주차장명  ev_충전소_500m
77         잠실역 공영주차장(시)          410
94      파미에(반포천) 주차장(시)          407
52      삼각지고가밑 공영주차장(시)          350
23       당산고가밑 공영주차장(시)          337
55  성내역(잠실나루역) 공영주차장(시)          336
14      남대문 화물 공영주차장(시)          312
69         옥수역 공영주차장(시)          304
32        동국제강 공영주차장(시)          301
24        당산노외 공영주차장(시)          295
53     서울글로벌센터 공영주차장(시)          294


잠실역이 410개로 1위

In [32]:
parking_merged.to_csv(
    r'C:\Users\seonu\Documents\DR-project\NEW\OUTPUT\서울시_공영주차장_축3.csv',
    index=False,
    encoding='utf-8-sig'
)
print("저장 완료")

# 축3 변수 현황 확인
print(parking_merged[['주차장명', '총 주차면', '평일운영시간', '주말운영시간', 'ev_충전소_500m']].describe())

저장 완료
             총 주차면      평일운영시간      주말운영시간  ev_충전소_500m
count    95.000000  100.000000  100.000000    100.00000
mean    160.505263   17.170000   16.400000    136.74000
std     308.781403    6.716563    8.137704     94.70663
min       1.000000    9.000000    5.000000      2.00000
25%       1.000000   10.000000    6.000000     59.75000
50%      38.000000   14.000000   24.000000    113.50000
75%     136.000000   24.000000   24.000000    188.25000
max    1431.000000   24.000000   24.000000    410.00000


- 주차면 5개 결측
- 주차면 min이 1?

In [33]:
# 결측 확인
print("총 주차면 결측:")
print(parking_merged[parking_merged['총 주차면'].isna()]['주차장명'].tolist())

# 1면 이하 확인
print("\n주차면 1면 이하:")
print(parking_merged[parking_merged['총 주차면'] <= 1][['주차장명', '총 주차면']].to_string())

총 주차면 결측:
['방화역 공영주차장(시)', '청계2 공영주차장(시)', '청계3 공영주차장(시)', '청계5 공영주차장(시)', '청계6 공영주차장(시)']

주차면 1면 이하:
                        주차장명  총 주차면
0    DDP동측(양쪽) 관광버스전용 주차장(시)    1.0
1            가마산고가밑 공영주차장(시)    1.0
2           가산동 금천교 공영주차장(시)    1.0
8               관철동 공영주차장(시)    1.0
11           난지중앙로노상공영주차장(시)    1.0
12      남대문 시장 관광버스전용 주차장(시)    1.0
13      남대문 초입 관광버스전용 주차장(시)    1.0
14           남대문 화물 공영주차장(시)    1.0
16    남산공원 소월로 관광버스전용 주차장(시)    1.0
17    남산공원 소파로 관광버스전용 주차장(시)    1.0
19           남산케이블카 공영주차장(시)    1.0
20            남산파출소 공영주차장(시)    1.0
22             당고개위 공영주차장(시)    1.0
23            당산고가밑 공영주차장(시)    1.0
26              대림역 공영주차장(시)    1.0
29          독산동 금천교 공영주차장(시)    1.0
30             돈화문로 공영주차장(시)    1.0
34    동순라길(종묘) 관광버스전용 주차장(시)    1.0
37          동호대교(남) 공영주차장(시)    1.0
38             마른내길 공영주차장(시)    1.0
39             마른내로 공영주차장(시)    1.0
43             문래1동 공영주차장(시)    1.0
44              방화로 공영주차장(시)    1.0
46             배오개길 공영주차장(시)    1

원본에서 중복 제거할 때 drop_duplicates로 첫 번째 행만 가져왔는데, 노상주차장은 원본에서 면단위로 행이 쪼개져 있어서 1면씩 등록된 거임

=> 근데 전처리에서 그 1면씩 등록된 주차장들을 위도/경도 중앙값 하나로 합쳤기 떄문에 흩어져 있어도 같은 이름으로 관리되는 하나의 주차장으로 봄

=> 따라서 실제 면수도 일관성 있게 원본에서 주차장명별로 합산해야함

In [34]:
# 주차장명별 총 주차면 합산
face_sum = parking_raw.groupby('주차장명')['총 주차면'].sum().reset_index()
face_sum.columns = ['주차장명', '총주차면_합산']

# 기존 컬럼 대체
parking_merged = parking_merged.drop(columns=['총 주차면'])
parking_merged = parking_merged.merge(face_sum, on='주차장명', how='left')

print(parking_merged[['주차장명', '총주차면_합산']].sort_values('총주차면_합산', ascending=False).head(10).to_string())
print(f"\n결측: {parking_merged['총주차면_합산'].isna().sum()}개")
print(f"min: {parking_merged['총주차면_합산'].min()}, max: {parking_merged['총주차면_합산'].max()}")

               주차장명  총주차면_합산
85     천호역 공영주차장(시)   1431.0
81   종묘주차장 공영주차장(시)   1317.0
56     세종로 공영주차장(시)   1260.0
94  파미에(반포천) 주차장(시)   1186.0
33     동대문 공영주차장(시)   1092.0
98   훈련원공원 공영주차장(시)    873.0
58    수서역북 공영주차장(시)    575.0
66      영남 공영주차장(시)    570.0
40   면목유수지 공영주차장(시)    567.0
71  용산주차빌딩 공영주차장(시)    561.0

결측: 5개
min: 2.0, max: 1431.0


결측 5개는 청계2,3,5,6,방화역

In [35]:
# 결측 5개 수동 합산
missing_mapping = {
    '방화역 공영주차장(시)': ['방화역(서) 공영주차장(시)', '방화역(동) 공영주차장(시)'],
    '청계2 공영주차장(시)': ['청계2(북2) 공영주차장(시)', '청계2(돈화문로) 공영주차장(시)'],
    '청계3 공영주차장(시)': ['청계3(북1) 공영주차장(시)', '청계3(남) 공영주차장(시)', '청계3(동호로) 공영주차장(시)'],
    '청계5 공영주차장(시)': ['청계5(북1) 공영주차장(시)', '청계5(방산동) 공영주차장(시)', '청계5(동호로_남) 공영주차장(시)', '청계5가 공영주차장(시)'],
    '청계6 공영주차장(시)': ['청계6(청평화) 공영주차장(시)', '청계6(북) 공영주차장(시)', '청계6(동평화) 공영주차장(시)'],
}

for our_name, raw_names in missing_mapping.items():
    total = parking_raw[parking_raw['주차장명'].isin(raw_names)]['총 주차면'].sum()
    parking_merged.loc[parking_merged['주차장명'] == our_name, '총주차면_합산'] = total
    print(f"{our_name}: {total}면")

print(f"\n결측 남은 수: {parking_merged['총주차면_합산'].isna().sum()}")

방화역 공영주차장(시): 135.0면
청계2 공영주차장(시): 19.0면
청계3 공영주차장(시): 56.0면
청계5 공영주차장(시): 107.0면
청계6 공영주차장(시): 53.0면

결측 남은 수: 0


In [36]:
parking_merged.to_csv(
    r'C:\Users\seonu\Documents\DR-project\NEW\OUTPUT\서울시_공영주차장_축3.csv',
    index=False,
    encoding='utf-8-sig'
)

# 축3 변수 최종 확인
print(parking_merged[['주차장명', '총주차면_합산', '평일운영시간', '주말운영시간', 'ev_충전소_500m']].describe())
print(f"\n결측 확인:")
print(parking_merged[['총주차면_합산', '평일운영시간', '주말운영시간', 'ev_충전소_500m']].isna().sum())

           총주차면_합산      평일운영시간      주말운영시간  ev_충전소_500m
count   100.000000  100.000000  100.000000    100.00000
mean    165.550000   17.170000   16.400000    136.74000
std     297.144198    6.716563    8.137704     94.70663
min       2.000000    9.000000    5.000000      2.00000
25%      18.000000   10.000000    6.000000     59.75000
50%      54.000000   14.000000   24.000000    113.50000
75%     135.000000   24.000000   24.000000    188.25000
max    1431.000000   24.000000   24.000000    410.00000

결측 확인:
총주차면_합산        0
평일운영시간         0
주말운영시간         0
ev_충전소_500m    0
dtype: int64


#### 서울시 자치구별 연료별 자동차 등록 현황 (OA-15640)

In [37]:
import pandas as pd

df_car = pd.read_csv(
    r'C:\Users\seonu\Documents\DR-project\NEW\dataset\서울시 자치구별 연료별 차종별 용도별 등록현황(26년3월).csv',
    encoding='cp949'
)
print(df_car.shape)
print(df_car.head(10).to_string())
print(df_car.columns.tolist())

(585, 14)
   그룹  그룹2  그룹3   시군구_코드 시군구명 연료_코드   연료명  용도별_코드   용도별       승용     승합      화물     특수        계
0   1    1    1      NaN  합 계   NaN     계     NaN     계  2770212  79389  296086  11753  3157440
1   0    0    0  11110.0  종로구     j   CNG     2.0  비사업용        3      6      13      0       22
2   0    0    0  11110.0  종로구     j   CNG     1.0   사업용        0     69       0      0       69
3   0    0    0  11110.0  종로구     b    경유     2.0  비사업용     8149   2497    3123    150    13919
4   0    0    0  11110.0  종로구     b    경유     1.0   사업용        3    140    1302     99     1544
5   0    0    0  11110.0  종로구     z  기타연료     2.0  비사업용        1     28      67     36      132
6   0    0    0  11110.0  종로구     z  기타연료     1.0   사업용        0      0     130      0      130
7   0    0    0  11110.0  종로구     q    수소     2.0  비사업용       85     13       0      0       98
8   0    0    0  11110.0  종로구     r  수소전기     2.0  비사업용       16      0       0      0       16
9   0    0    0  11110.0  종로구 

전기차만 필터링

In [38]:
# 전기차만 필터링 (연료명 = '전기')
ev_by_gu = df_car[
    (df_car['연료명'] == '전기') & 
    (df_car['시군구명'] != '합 계')
].groupby('시군구명')['계'].sum().reset_index()

ev_by_gu.columns = ['자치구', 'ev_등록대수']
ev_by_gu = ev_by_gu.sort_values('ev_등록대수', ascending=False)

print(ev_by_gu.to_string())

     자치구  ev_등록대수
0    강남구    15468
14   서초구     8665
3    강서구     8009
17   송파구     7851
1    강동구     5653
15   성동구     4331
19  영등포구     4323
12   마포구     4083
18   양천구     3941
8    노원구     3918
21   은평구     3656
6    구로구     3621
16   성북구     3533
24   중랑구     3158
10  동대문구     3021
4    관악구     3005
11   동작구     2867
13  서대문구     2799
20   용산구     2674
9    도봉구     2535
5    광진구     2445
7    금천구     2257
23    중구     2000
22   종로구     1896
2    강북구     1856


강남구 압도적 1위 

In [40]:
print(parking_merged.columns.tolist())

['주차장명', '주차장종류', '주차장종류명', '위도', '경도', '원본명칭_수', '전체_행_수', '평일운영시간', '주말운영시간', 'ev_충전소_500m', '총주차면_합산']


주차장 원본에서 주소 가져오기

In [42]:
# 원본에서 주차장명 + 주소 가져오기
addr_map = parking_raw[['주차장명', '주소']].drop_duplicates(subset='주차장명', keep='first')

parking_merged = parking_merged.merge(addr_map, on='주차장명', how='left')

# 자치구 추출
parking_merged['자치구'] = parking_merged['주소'].str.extract(r'(\S+구)\s')

print(parking_merged[['주차장명', '주소', '자치구']].head(5).to_string())
print(f"자치구 추출 실패: {parking_merged['자치구'].isna().sum()}개")

                      주차장명               주소  자치구
0  DDP동측(양쪽) 관광버스전용 주차장(시)    중구 을지로7가 2-36   중구
1          가마산고가밑 공영주차장(시)   구로구 구로동 414-13  구로구
2         가산동 금천교 공영주차장(시)    금천구 가산동 691-3  금천구
3            가양3동 공영주차장(시)  강서구 가양동 1488-12  강서구
4           가양라이품 공영주차장(시)   강서구 가양동 1457-1  강서구
자치구 추출 실패: 5개


결측확인

원본에서 drop_duplicates로 첫 번째 행만 가져왔는데 저 5개는 원본에 다른 이름으로 등록돼 있어서 주소가 NaN으로 온 거임

In [43]:
print(parking_merged[parking_merged['자치구'].isna()][['주차장명', '주소']].to_string())

            주차장명   주소
45  방화역 공영주차장(시)  NaN
87  청계2 공영주차장(시)  NaN
88  청계3 공영주차장(시)  NaN
89  청계5 공영주차장(시)  NaN
90  청계6 공영주차장(시)  NaN


수동매핑

In [49]:
for keyword in ['청계5', '청계6']:
    sub = parking_raw[parking_raw['주차장명'].str.contains(keyword, na=False)][['주차장명', '주소']].drop_duplicates()
    print(sub.head(3).to_string())
    print()

                     주차장명              주소
624      청계5(북1) 공영주차장(시)  종로구 종로5가 458-2
649   청계5(동호로_북) 공영주차장(시)  종로구 종로5가 458-2
1150  청계5(동호로_남) 공영주차장(시)   중구 방산동 4-47 0

                      주차장명               주소
191      청계6(청평화) 공영주차장(시)  중구 신당동 217-91 0
661        청계6(북) 공영주차장(시)  종로구 종로6가 289-49
1222  청계6(신평화시장앞) 공영주차장(시)  중구 신당동 217-91 0



청계5, 6 둘 다 종로구/중구 걸쳐있음 하...

면수 기준 가중 자치구 — 종로구 면수 vs 중구 면수 비교해서 더 많은 쪽으로 배정

In [50]:
for keyword, our_name in [('청계5', '청계5 공영주차장(시)'), ('청계6', '청계6 공영주차장(시)')]:
    sub = parking_raw[parking_raw['주차장명'].str.contains(keyword, na=False)][['주차장명', '주소', '총 주차면']]
    sub['자치구'] = sub['주소'].str.extract(r'(\S+구)\s')
    gu_sum = sub.groupby('자치구')['총 주차면'].sum()
    best_gu = gu_sum.idxmax()
    print(f"{our_name} → {gu_sum.to_dict()} → 배정: {best_gu}")

청계5 공영주차장(시) → {'종로구': 31.0, '중구': 87.0} → 배정: 중구
청계6 공영주차장(시) → {'종로구': 39.0, '중구': 22.0} → 배정: 종로구


In [55]:
manual_gu = {
    '방화역 공영주차장(시)': '강서구',
    '청계2 공영주차장(시)': '종로구',
    '청계3 공영주차장(시)': '종로구',
    '청계5 공영주차장(시)': '중구',
    '청계6 공영주차장(시)': '종로구',
}

for name, gu in manual_gu.items():
    parking_merged.loc[parking_merged['주차장명'] == name, '자치구'] = gu

print(f"결측 남은 수: {parking_merged['자치구'].isna().sum()}")

# EV 등록대수 조인
# 중복 컬럼 제거 후 재조인
if 'ev_등록대수' in parking_merged.columns:
    parking_merged = parking_merged.drop(columns=['ev_등록대수'])

parking_merged = parking_merged.merge(ev_by_gu, on='자치구', how='left')
print(f"결측: {parking_merged['ev_등록대수'].isna().sum()}")
print(parking_merged[['주차장명', '자치구', 'ev_등록대수']].head(10).to_string())

결측 남은 수: 0
결측: 0
                      주차장명  자치구  ev_등록대수
0  DDP동측(양쪽) 관광버스전용 주차장(시)   중구     2000
1          가마산고가밑 공영주차장(시)  구로구     3621
2         가산동 금천교 공영주차장(시)  금천구     2257
3            가양3동 공영주차장(시)  강서구     8009
4           가양라이품 공영주차장(시)  강서구     8009
5             가양역 공영주차장(시)  강서구     8009
6            개화산역 공영주차장(시)  강서구     8009
7             개화역 공영주차장(시)  강서구     8009
8             관철동 공영주차장(시)  종로구     1896
9        구로디지털단지역 공영주차장(시)  구로구     3621


In [56]:
parking_merged.to_csv(
    r'C:\Users\seonu\Documents\DR-project\NEW\OUTPUT\서울시_공영주차장_축3.csv',
    index=False,
    encoding='utf-8-sig'
)
print("저장 완료")
print(f"\n축3 최종 변수 목록:")
print(parking_merged[['주차장명', '총주차면_합산', '평일운영시간', '주말운영시간', 'ev_충전소_500m', 'ev_등록대수']].describe())

저장 완료

축3 최종 변수 목록:
           총주차면_합산      평일운영시간      주말운영시간  ev_충전소_500m      ev_등록대수
count   100.000000  100.000000  100.000000    100.00000    100.00000
mean    165.550000   17.170000   16.400000    136.74000   4504.33000
std     297.144198    6.716563    8.137704     94.70663   3632.92008
min       2.000000    9.000000    5.000000      2.00000   1896.00000
25%      18.000000   10.000000    6.000000     59.75000   2000.00000
50%      54.000000   14.000000   24.000000    113.50000   3021.00000
75%     135.000000   24.000000   24.000000    188.25000   4325.00000
max    1431.000000   24.000000   24.000000    410.00000  15468.00000


변수5개합치기

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
# 데이터 로드
parking_merged = pd.read_csv(
    r'C:\Users\seonu\Documents\DR-project\NEW\OUTPUT\서울시_공영주차장_축3.csv',
    encoding='utf-8-sig'
)

print(parking_merged.shape)
print(parking_merged.columns.tolist())

(100, 16)
['주차장명', '주차장종류', '주차장종류명', '위도', '경도', '원본명칭_수', '전체_행_수', '평일운영시간', '주말운영시간', 'ev_충전소_500m', '총주차면_합산', '주소', '자치구', 'ev_등록대수_x', 'ev_등록대수_y', 'ev_등록대수']


헉 중복저장됨~~

In [3]:
# ev_등록대수 컬럼 3개 확인
print(parking_merged[['ev_등록대수_x', 'ev_등록대수_y', 'ev_등록대수']].head(5))
print(parking_merged[['ev_등록대수_x', 'ev_등록대수_y', 'ev_등록대수']].isna().sum())

   ev_등록대수_x  ev_등록대수_y  ev_등록대수
0       2000       2000     2000
1       3621       3621     3621
2       2257       2257     2257
3       8009       8009     8009
4       8009       8009     8009
ev_등록대수_x    0
ev_등록대수_y    0
ev_등록대수      0
dtype: int64


In [4]:
# 중복 컬럼 제거
parking_merged = parking_merged.drop(columns=['ev_등록대수_x', 'ev_등록대수_y'])

# 축3 점수화
scaler = MinMaxScaler()

# 자체소비 그룹 정규화
self_cols = ['총주차면_합산', '평일운영시간', '주말운영시간']
parking_merged[['총주차면_norm', '평일운영_norm', '주말운영_norm']] = scaler.fit_transform(parking_merged[self_cols])

# EV충전 그룹 정규화 (ev_충전소_500m 역방향)
parking_merged['ev_충전소_역'] = parking_merged['ev_충전소_500m'].max() - parking_merged['ev_충전소_500m']
ev_cols = ['ev_등록대수', 'ev_충전소_역']
parking_merged[['ev_등록_norm', 'ev_충전소_역_norm']] = scaler.fit_transform(parking_merged[ev_cols])

# 그룹별 점수
parking_merged['자체소비_점수'] = (parking_merged['총주차면_norm'] + parking_merged['평일운영_norm'] + parking_merged['주말운영_norm']) / 3
parking_merged['EV충전_점수'] = (parking_merged['ev_등록_norm'] + parking_merged['ev_충전소_역_norm']) / 2

# 축3 최종 점수
parking_merged['축3_자가소비적합성'] = (parking_merged['자체소비_점수'] + parking_merged['EV충전_점수']) / 2

print(parking_merged[['주차장명', '자체소비_점수', 'EV충전_점수', '축3_자가소비적합성']]
      .sort_values('축3_자가소비적합성', ascending=False)
      .head(15).to_string())

               주차장명   자체소비_점수   EV충전_점수  축3_자가소비적합성
58    수서역북 공영주차장(시)  0.800327  0.895833    0.848080
95     학여울 공영주차장(시)  0.708188  0.943627    0.825907
76    일원터널 공영주차장(시)  0.668766  0.930147    0.799457
85     천호역 공영주차장(시)  1.000000  0.443557    0.721779
7      개화역 공영주차장(시)  0.778866  0.662706    0.720786
49     복정역 공영주차장(시)  0.750641  0.675268    0.712955
81   종묘주차장 공영주차장(시)  0.973408  0.444853    0.709130
63   신천유수지 공영주차장(시)  0.710987  0.637278    0.674132
51    사당노외 공영주차장(시)  0.708887  0.624374    0.666631
6     개화산역 공영주차장(시)  0.740844  0.583049    0.661947
98   훈련원공원 공영주차장(시)  0.869839  0.399665    0.634752
4    가양라이품 공영주차장(시)  0.675064  0.589177    0.632121
33     동대문 공영주차장(시)  0.920924  0.343292    0.632108
40   면목유수지 공영주차장(시)  0.798460  0.398208    0.598334
94  파미에(반포천) 주차장(시)  0.942850  0.253050    0.597950


인사이트

- 수서역북이 1위 — 면수 크고 운영시간 길면서 EV 충전 공백도 높아서 두 그룹 모두 균형있게 높음

- 천호역, 종묘, 동대문, 파미에 — 자체소비 점수는 높은데 EV충전 점수가 낮음. 규모는 크지만 주변에 이미 충전 인프라가 많은 지역이라는 뜻.

- 학여울, 일원터널 — EV충전 점수가 높은데 자체소비가 상대적으로 낮음. EV 수요 공백 지역인데 주차장 규모가 작은 케이스.

In [6]:
# 정리해서 저장
parking_merged.to_csv(
    r'C:\Users\seonu\Documents\DR-project\NEW\OUTPUT\서울시_공영주차장_축3.csv',
    index=False,
    encoding='utf-8-sig'
)
print("저장 완료")
print(f"\n축3 점수 분포:")
print(parking_merged['축3_자가소비적합성'].describe())

저장 완료

축3 점수 분포:
count    100.000000
mean       0.425338
std        0.187192
min        0.072612
25%        0.250449
50%        0.451729
75%        0.569683
max        0.848080
Name: 축3_자가소비적합성, dtype: float64


In [7]:
print(parking_merged.columns.tolist())

['주차장명', '주차장종류', '주차장종류명', '위도', '경도', '원본명칭_수', '전체_행_수', '평일운영시간', '주말운영시간', 'ev_충전소_500m', '총주차면_합산', '주소', '자치구', 'ev_등록대수', '총주차면_norm', '평일운영_norm', '주말운영_norm', 'ev_충전소_역', 'ev_등록_norm', 'ev_충전소_역_norm', '자체소비_점수', 'EV충전_점수', '축3_자가소비적합성']


두 가지 파일로 나눠서 저장

In [8]:
# PCA용 축3 점수만
cols_for_pca = ['주차장명', '위도', '경도', '축3_자가소비적합성']
parking_merged[cols_for_pca].to_csv(
    r'C:\Users\seonu\Documents\DR-project\NEW\OUTPUT\축3_점수.csv',
    index=False, encoding='utf-8-sig'
)

# 전체 분석 결과 보존
parking_merged.to_csv(
    r'C:\Users\seonu\Documents\DR-project\NEW\OUTPUT\서울시_공영주차장_축3.csv',
    index=False, encoding='utf-8-sig'
)
print("저장 완료")

저장 완료
